<a href="https://colab.research.google.com/github/tanishjain-tmj/Data-Cleanser-datapreprocessing/blob/main/Data_cleanser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Project :Data cleanser**

In [51]:
import pandas as pd
import numpy as np

In [52]:
df=pd.read_csv("/content/patient_health_records.csv")
df.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,1001,59.0,Female,West,22.0,137.2,205.9,90.4,0
1,1002,50.0,Female,North,17.3,143.6,243.6,92.1,0
2,1003,62.0,Male,West,28.7,112.8,273.2,118.7,0
3,1004,NaN,Female,North,NaN,145.1,275.1,115.2,0
4,1005,NaN,Female,South,27.0,124.3,217.0,92.1,0


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      500 non-null    int64  
 1   age             470 non-null    float64
 2   gender          475 non-null    object 
 3   region          475 non-null    object 
 4   bmi             460 non-null    float64
 5   blood_pressure  500 non-null    float64
 6   cholesterol     465 non-null    float64
 7   glucose         465 non-null    float64
 8   disease_risk    500 non-null    int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 35.3+ KB


In [54]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
patient_id,500.0,1250.500000,144.481833,1001.0,1125.750,1250.50,1375.250,1500.0
age,470.0,52.114894,14.074414,18.0,42.000,52.00,61.000,85.0
bmi,460.0,27.484130,5.379505,7.9,24.300,27.25,30.325,65.0
blood_pressure,500.0,128.943400,17.461264,48.0,118.575,128.30,138.325,250.0
cholesterol,465.0,220.664301,33.235677,35.0,201.300,219.10,240.000,420.0
glucose,465.0,102.403011,31.121859,18.0,87.300,100.90,115.500,420.0
disease_risk,500.0,0.098000,0.297612,0.0,0.000,0.00,0.000,1.0


**Part A:**

In [55]:
#Part A Q1:
df.isnull().sum()

,0
patient_id,0
age,30
gender,25
region,25
bmi,40
blood_pressure,0
cholesterol,35
glucose,35
disease_risk,0


In [56]:
missing_report = pd.DataFrame({"Missing Count": df.isnull().sum(),"Missing Percentage": (df.isnull().mean()*100).round(2)})
missing_report

,Missing Count,Missing Percentage
patient_id,0,0.0
age,30,6.0
gender,25,5.0
region,25,5.0
bmi,40,8.0
blood_pressure,0,0.0
cholesterol,35,7.0
glucose,35,7.0
disease_risk,0,0.0


Missing value:

The dataset contains missing values in columns age,gender , region,bmi,cholestrol and glucose .
missing value percentage is betweeen 5-8% .

In [57]:
#part A Q2:
#simple imputer (numerical)
from sklearn.impute import SimpleImputer

imputer=SimpleImputer(strategy="mean")
df[["bmi"]]=imputer.fit_transform(df[["bmi"]])
print("missing bmi after mean imputation:")
print(df['bmi'].isnull().sum())

missing bmi after mean imputation:
0


In [58]:
#simple imputer(categorical)
imputer=SimpleImputer(strategy="most_frequent")
df[["region"]]=imputer.fit_transform(df[["region"]])
print("Missing value in region after impution:",df['region'].isnull().sum())

#most frequent imputer
df[["gender"]]=imputer.fit_transform(df[["gender"]])
print("Missing value in gender after impution:",df['gender'].isnull().sum())


Missing value in region after impution: 0
Missing value in gender after impution: 0


In [59]:
#Missing Indicator + Random Sample Imputation:
columns=["age"]
for col in columns:
  df[col+ "_missing"]=df[col].isnull().astype(int)

def random_simple_imputation(df,column):
  missing= df[column].isnull()
  observed_values=df.loc[~missing,column]
  random_values=np.random.choice(observed_values,size=missing.sum(),replace=True)
  df.loc[missing,column]=random_values
  return df

np.random.seed(42)
for col in columns:
  df=random_simple_imputation(df,col)

In [60]:
#knn imputer:
from sklearn.impute import KNNImputer
columns=["cholesterol"]
knn_imputer=KNNImputer(n_neighbors=5)
df[columns]=knn_imputer.fit_transform(df[columns])
df[columns].isnull().sum()

,0
cholesterol,0


In [61]:
#mice:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
mice_imputer=IterativeImputer(max_iter=10,random_state=42)
df[["glucose"]]=mice_imputer.fit_transform(df[["glucose"]])
df['glucose'].isnull().sum()

np.int64(0)

**Part B:**

In [62]:
print("Before removing Outliers:")
df[["cholesterol", "glucose"]].describe().T

,count,mean,std,min,25%,50%,75%,max
cholesterol,500.0,220.664301,32.048909,35.0,203.150,220.664301,238.475,420.0
glucose,500.0,102.403011,30.010570,18.0,89.025,102.403011,114.825,420.0


In [63]:
#Part B Q3:
#zscore:
from scipy.stats import zscore
z_columns=["cholesterol","glucose"]
for col in z_columns:
  z=np.abs(zscore(df[col],nan_policy="omit"))
  df=df[z<3]

In [64]:
print("After removing Outliers:")
df[["cholesterol", "glucose"]].describe().T

,count,mean,std,min,25%,50%,75%,max
cholesterol,492.0,220.323273,26.340092,143.7,203.275,220.664301,238.15,280.0
glucose,492.0,100.503669,20.144009,18.0,88.450,102.401505,114.00,174.4


In [65]:
print("Before removing Outliers:")
df['bmi'].describe()

,bmi
count,492.000000
mean,27.462595
std,5.164449
min,7.900000
25%,24.500000
50%,27.484130
75%,30.100000
max,65.000000


In [66]:
#IQR:
q1=df['bmi'].quantile(0.25)
q3=df['bmi'].quantile(0.75)
iqr=q3-q1
lower=q1-1.58*iqr
upper=q3+1.58*iqr
df=df[(df['bmi']>=lower)&(df['bmi']<=upper)]

In [67]:
print("After removing Outliers:")
df['bmi'].describe()

,bmi
count,486.000000
mean,27.302257
std,4.216721
min,16.000000
25%,24.525000
50%,27.484130
75%,30.000000
max,38.700000


In [68]:
print("Before removing Outliers:")
df['age'].describe()

,age
count,486.000000
mean,52.405350
std,14.193301
min,18.000000
25%,42.000000
50%,52.000000
75%,62.000000
max,85.000000


In [69]:
#percentile method
lower=df['age'].quantile(0.01)
upper=df['age'].quantile(0.99)
df=df[(df['age']>=lower)&(df['age']<=upper)]

In [70]:
print("After removing Outliers:")
df['age'].describe()

,age
count,482.000000
mean,52.682573
std,13.919761
min,21.000000
25%,42.250000
50%,53.000000
75%,62.000000
max,85.000000


In [71]:
print("Before removing Outliers:")
df['blood_pressure'].describe()

,blood_pressure
count,482.000000
mean,128.935477
std,17.604471
min,48.000000
25%,118.500000
50%,128.300000
75%,138.475000
max,250.000000


In [72]:
#Q4:Winsorization
from scipy.stats.mstats import winsorize
df["blood_pressure"]=winsorize(df["blood_pressure"],limits=[0.01,0.01])


In [73]:
print("After removing Outliers:")
df['blood_pressure'].describe()

/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:4809: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


,blood_pressure
count,482.000000
mean,128.618672
std,14.675702
min,92.200000
25%,118.500000
50%,128.300000
75%,138.475000
max,165.900000


In [77]:
df.shape

(482, 10)

#Q5:
previously the data shape was (500,10) and now after removing or imputing all outliers the data shape is (482,10) , outliers were present in all the numerical columns .

**part C:**

In [79]:
#Q6:
df = df.reset_index(drop=True)
print("cleaned data:",df)

cleaned data:      patient_id   age  gender region       bmi  blood_pressure  cholesterol  \
0          1001  59.0  Female   West  22.00000           137.2        205.9   
1          1002  50.0  Female  North  17.30000           143.6        243.6   
2          1003  62.0    Male   West  28.70000           112.8        273.2   
3          1004  85.0  Female  North  27.48413           145.1        275.1   
4          1005  50.0  Female  South  27.00000           124.3        217.0   
..          ...   ...     ...    ...       ...             ...          ...   
477        1496  60.0  Female  South  30.40000           122.2        203.3   
478        1497  36.0  Female  North  27.40000           112.8        191.7   
479        1498  49.0  Female   West  20.40000           132.4        201.1   
480        1499  39.0    Male   East  25.60000           129.8        185.8   
481        1500  31.0  Female  South  23.60000           112.1        163.3   

     glucose  disease_risk  age_missi

#Q7:
a. KNN and MICE imputation method are best strategy compare to all as they are less effecting the actual dataset .

b. percentile and winsorization methods are best for treating outliers as they prevent the data quality in best way.

c.data cleaning improve dataset usability in a way that after cleaning the dataset it will give as accurate and better output when used in ml models , it improves the accuracy rate.